# 02 - Files and GitHub (Python)

**File:** `notebooks/02_files_and_github_python.ipynb`

**What this does:** Shows how a notebook finds folders on disk, writes a file, reads it back, and how that work gets to GitHub.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **Python 3**, then choose *Run > Run All Cells*.

**Inputs:** none -- this notebook creates the file it reads

**Outputs:** `outputs/stations.csv` and `outputs/warm_stations.csv`

## Where am I? Finding the repo folder

A notebook runs from the folder it lives in (`notebooks/`), **not** from the top
of the repository. That trips people up constantly: a path that works in a
terminal at the repo root will not work here.

Rather than writing `../` everywhere, the cell below works out where the top of
the repo is once, and builds paths from there. Every notebook here uses this
same short block.

In [ ]:
from pathlib import Path

# Path.cwd() is the folder this notebook runs in.
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

print("Repo folder:", REPO)

## 1. What files are around me?

`iterdir()` lists the contents of a folder. This is the notebook equivalent of
typing `ls` in a terminal.

In [ ]:
for path in sorted(REPO.iterdir()):
    if path.name.startswith("."):
        continue          # hide .git and friends
    kind = "folder" if path.is_dir() else "file"
    print(f"{kind:7s} {path.name}")

## 2. Writing a file

Results go in an `outputs/` folder. Keeping outputs separate from inputs means a
mistake never destroys your original data -- a habit worth forming early.

Here we build a small table and save it as a CSV.

In [ ]:
import pandas as pd

stations = pd.DataFrame({
    "station_id":   ["ST-001", "ST-002", "ST-003", "ST-004", "ST-005"],
    "name":         ["Kaena Point", "Penguin Bank", "Makapuu",
                     "Waianae Deep", "Kaiwi Channel"],
    "latitude":     [21.5760, 21.0400, 21.3100, 21.4200, 21.2600],
    "longitude":    [-158.2800, -157.3900, -157.6500, -158.3600, -157.7300],
    "water_temp_c": [25.4, 26.1, 24.8, 25.9, 26.4],
})

outputs = REPO / "outputs"
outputs.mkdir(exist_ok=True)          # makes the folder, fine if it already exists

csv_path = outputs / "stations.csv"
stations.to_csv(csv_path, index=False)   # index=False keeps the row numbers out

print("Wrote", csv_path)
print(csv_path.stat().st_size, "bytes")

## 3. Reading it back

`read_csv` turns the file back into a table (a "DataFrame"). This is the same
call you would use on any CSV, wherever it came from.

In [ ]:
loaded = pd.read_csv(csv_path)

print(f"{len(loaded)} rows, {len(loaded.columns)} columns")
loaded.head()

## 4. Doing something with it

Keep only the warmer stations. Nothing here changes the file on disk -- `loaded`
is a copy held in memory.

In [ ]:
warm = loaded[loaded["water_temp_c"] > 25.5]

print(f"{len(warm)} of {len(loaded)} stations are warmer than 25.5 C")
warm[["station_id", "name", "water_temp_c"]]

## 5. Saving the result

Write it alongside the first file, under a different name.

In [ ]:
result_path = outputs / "warm_stations.csv"
warm.to_csv(result_path, index=False)

print("Wrote", result_path)

## 6. Getting your work back to GitHub

A notebook can run shell commands by starting a line with `!`. So you can check
on git without leaving Jupyter:

In [ ]:
!git status --short

Notice what is **not** there: the files you just wrote to `outputs/`. That folder
is listed in [`.gitignore`](../.gitignore), so git ignores it on purpose --
results are something you can always regenerate by re-running the notebook, and
data files are usually too big for a repository anyway.

What you probably *do* see is this notebook itself marked `M` for modified. You
only ran it -- but running a notebook stores its output inside the file, so the
file really did change.

To save your work back to GitHub, run these in a terminal
(*File > New > Terminal* in JupyterLab):

```bash
git checkout -b your-name-your-feature    # once, before you start
git add .
git commit -m "a short note about what you did"
git push -u origin your-name-your-feature
```

Everyone works on their own branch here -- no forking. The full walkthrough is in
the [main README](../README.md).

Two things worth knowing:

- **Notebooks produce noisy diffs.** A notebook stores its outputs inside the
  file, so re-running it shows up as a change even when you edited nothing.
  Running *Kernel > Restart Kernel and Clear Outputs* before committing keeps
  pull requests readable.
- **Keep data files out of git.** They are usually too big, and GitHub rejects
  anything over 100 MB. See [`.gitignore`](../.gitignore).

## Done

Next: **`03_aquaview_stac_python.ipynb`**, which pulls real data off the internet.